# Pipeline de extracción de ratios financieros — Excel

Notebook correspondiente al **experimento de extracción desde Excel** descrito en la sección 4.2 de la memoria del TFM.

El objetivo es extraer las 22 partidas contables necesarias para el cálculo de ratios financieros a partir de un fichero Excel con estados financieros, con independencia del formato concreto del fichero: layout vertical u horizontal, multi-hoja, con celdas combinadas, en español o inglés, con o sin columnas de variación interanual intercaladas.

## Empresa de prueba
**Grifols, S.A.** — exportación SABI del balance consolidado y cuenta de pérdidas y ganancias para los ejercicios 2022, 2023 y 2024.

## Pipeline en 5 pasos

```
Excel (.xlsx)
    │
    ├─ Paso 1: Detección de layout          ← LLM (muestra serializada de filas/columnas)
    │
    ├─ Paso 2: DataFrame de conceptos       ← Python determinista
    │
    ├─ Paso 3: Selección de 22 partidas     ← LLM (índice posicional completo)
    │
    ├─ Paso 4: Extracción de valores        ← Python determinista
    │
    └─ Paso 5: Cálculo de 22 ratios         ← Python determinista

```

> **Principio de diseño:** todo lo que puede calcularse de forma exacta se calcula con código Python. El LLM interviene únicamente en los dos pasos que requieren razonamiento estructural: inferir cómo está organizado el fichero y localizar partidas por nombre dentro de un índice jerárquico.

---


## Celda 1 — Importaciones

Librerías necesarias: `pandas` para manipulación tabular, `json` y `re` para parsear respuestas del LLM, `openai` como cliente compatible con la API de Groq, y `httpx` para configurar timeouts en las llamadas HTTP.


In [11]:
# Imports base
import json
import re
import os
import time
import pandas as pd
import httpx

from openai import OpenAI

## Celda 2 — Configuración

Carga de credenciales desde `.env` (nunca hardcodeadas en el código). Se definen:
- **`MODELO1`** (`openai/gpt-oss-120b`): modelo principal para detección de layout y selección de partidas.
- **`MODELO2`** (`llama-3.3-70b-versatile`): modelo alternativo más ligero para el mapeo de partidas.
- **`RUTA_EXCEL`**: ruta al fichero de entrada — actualizar antes de ejecutar.

El bloque `if not GROQ_API_KEY` detiene la ejecución antes de llegar a ninguna llamada si la key no está disponible.


In [12]:
# Cargar .env si hace falta
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY  = os.getenv("GROQ_API_KEY_3", "")
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
MODELO1        = "openai/gpt-oss-120b"
MODELO2        = "llama-3.3-70b-versatile"

RUTA_EXCEL = r"C:\Users\david\Desktop\david\tfm\TFM PYTHON\inputs\balance_pyg.xlsx"

if not GROQ_API_KEY:
    raise ValueError("❌ Falta GROQ_API_KEY_1")
else:
    print("✅ API key cargada")

✅ API key cargada


## Celda 3 — Wrappers de llamada al LLM

`llamar_llm1` y `llamar_llm2` son wrappers sobre el cliente OpenAI apuntando a Groq. Ambas:
- Fijan `temperature=0` para garantizar salidas deterministas y reproducibles.
- Usan `httpx.Client(timeout=120)` para evitar cortes en llamadas con prompts largos (el prompt de selección de partidas puede superar los 2.000 tokens de entrada).
- Devuelven directamente el texto de la respuesta, sin parseo.


In [13]:
def llamar_llm1(prompt: str, system: str, temperatura: float = 0.0) -> str:
    client = OpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        http_client=httpx.Client(timeout=120),
    )

    resp = client.chat.completions.create(
        model=MODELO1,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
        temperature=temperatura,
        max_tokens=2048,
    )

    return resp.choices[0].message.content.strip()


def llamar_llm2(prompt: str, system: str, temperatura: float = 0.0) -> str:
    client = OpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        http_client=httpx.Client(timeout=120),
    )

    resp = client.chat.completions.create(
        model=MODELO2,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
        temperature=temperatura,
        max_tokens=2048,
    )

    return resp.choices[0].message.content.strip()

## Celda 4 — Lectura del Excel (multi-hoja)

`leer_excel_multi` carga todas las hojas del fichero sin asumir ninguna estructura previa:
- `header=None`: no se fija ninguna fila como cabecera, se lee todo como datos.
- Las columnas se renombran a `col_0`, `col_1`... para evitar que pandas interprete celdas combinadas como cabeceras reales.
- Devuelve `{nombre_hoja: (rows_como_lista_de_dicts, lista_de_col_ids)}`.

Este enfoque agnóstico es necesario porque el problema de las celdas combinadas en Excel hace que, al leer con pandas, los valores queden desplazados respecto a las columnas donde uno esperaría encontrarlos.


In [14]:
def leer_excel_multi(ruta: str) -> dict:
    xls = pd.ExcelFile(ruta)
    hojas = {}

    for nombre in xls.sheet_names:
        try:
            df = pd.read_excel(xls, sheet_name=nombre, header=None)
            df.columns = [f"col_{i}" for i in range(len(df.columns))]
            rows = df.to_dict(orient="records")
            hojas[nombre] = (rows, list(df.columns))
        except Exception as e:
            print(f"⚠️ Error en hoja '{nombre}': {e}")

    return hojas

## Celda 5 — Normalización de texto

`_norm` elimina tildes y convierte a minúsculas. Se usa exclusivamente para comparar el contenido de las celdas con las palabras clave financieras del filtro de hojas, sin afectar al texto que se envía al LLM ni al que se extrae como resultado.


In [15]:
def _norm(s: str) -> str:
    s = s.lower()
    for a, b in [("á","a"),("é","e"),("í","i"),("ó","o"),("ú","u"),("ü","u"),("ñ","n")]:
        s = s.replace(a, b)
    return s.strip()

## Celda 6 — Filtro de hojas financieras

`es_hoja_financiera` detecta si una hoja contiene contenido contable buscando palabras clave en español e inglés (`activo`, `pasivo`, `assets`, `liabilities`...) en las primeras 50 filas. Solo las hojas que acumulan al menos 5 coincidencias se pasan al pipeline.

Esto evita procesar hojas auxiliares (portadas, índices, hojas de metadatos) que acompañan frecuentemente a los ficheros exportados de SABI o Bloomberg y que romperían la detección de layout al no tener estructura tabular financiera.


In [16]:
KEYWORDS_FIN = [
    # Español
    "activo", "pasivo", "patrimonio", "ingresos", "gastos",
    "resultado", "beneficio", "perdida", "ventas",
    "deuda", "existencias", "efectivo",

    # Inglés
    "assets", "liabilities", "equity", "revenue",
    "expenses", "profit", "loss", "sales",
    "debt", "inventory", "cash"
]


def es_hoja_financiera(rows: list[dict], umbral: int = 5) -> bool:
    hits = 0

    for row in rows[:50]:
        for v in row.values():
            if v is None:
                continue
            s = _norm(str(v))
            if any(k in s for k in KEYWORDS_FIN):
                hits += 1
                if hits >= umbral:
                    return True

    return False

## Celda 7 — Serialización de la muestra para el LLM

`serializar_muestra` convierte las primeras N filas y M columnas del DataFrame en texto plano con el formato `fila=i, col=col_j: valor`, filtrando celdas vacías y NaN.

Este texto es lo que recibe el LLM para inferir el layout. Enviamos solo una muestra (por defecto 30×30) y nunca valores numéricos completos del balance, lo que mantiene el coste en tokens bajo y evita que el modelo intente leer datos en lugar de inferir estructura.


In [17]:
def serializar_muestra(rows, col_ids, n_filas=30, n_cols=30) -> str:
    cols_muestra = col_ids[:n_cols]
    lineas = []

    for i, row in enumerate(rows[:n_filas]):
        for col in cols_muestra:
            v = row.get(col)
            s = str(v).strip()

            if s and s.lower() not in ("nan", "none", "nat", ""):
                s_trunc = s[:80] if len(s) > 80 else s
                lineas.append(f"fila={i}, col={col}: {s_trunc}")

    return "\n".join(lineas)

## Celda 8 — System prompt para detección de layout

Define el contrato de salida del LLM: un JSON con:
- `layout_tipo`: `"vertical"` (conceptos en filas, lo más habitual en SABI) u `"horizontal"` (conceptos en columnas, habitual en Bloomberg o plantillas internas).
- `col_conceptos` / `fila_conceptos`: índice de la columna o fila donde están los nombres de las partidas.
- `cols_valores` / `filas_valores`: lista de índices de columnas o filas con valores numéricos.
- `periodos`: fechas de cierre de cada ejercicio detectadas en la cabecera.
- `unidades` y `multiplicador`: para convertir correctamente (miles, millones, euros).

El prompt contempla explícitamente la existencia de celdas combinadas y de columnas vacías intercaladas entre las de datos, que son los dos problemas más frecuentes en exportaciones reales.


In [18]:
SYSTEM_LAYOUT = """Eres un experto en hojas de cálculo financieras.

El Excel puede estar en:

1) VERTICAL → conceptos en filas
2) HORIZONTAL → conceptos en columnas

Puede haber celdas combinadas.

---

Detecta:

- tipo de layout
- dónde están conceptos
- dónde están valores
- periodos
- unidades

---

Devuelve SOLO este JSON:

{
  "layout_tipo": "vertical" | "horizontal",

  "col_conceptos": <int o null>,
  "fila_conceptos": <int o null>,

  "cols_valores": [int, ...],
  "filas_valores": [int, ...],

  "periodos": ["texto", ...],
  "unidades": "texto",
  "multiplicador": <numero>,

  "fila_datos_inicio": <int o null>,
  "col_datos_inicio": <int o null>,

  "notas": "texto breve"
}
"""

## Celda 9 — Paso 1: detección de layout mediante LLM

`detectar_layout` envía la muestra serializada al LLM y parsea la respuesta JSON. Si el modelo devuelve texto adicional antes o después del bloque JSON (lo que ocurre en algunos modelos pese a las instrucciones), se extrae el bloque `{...}` por posición de llaves en lugar de fallar.

Este es el único punto del pipeline donde se delega razonamiento estructural al modelo. El resultado es un diccionario que describe cómo está organizado el fichero, y a partir de aquí todo es Python determinista.


In [19]:
def detectar_layout(muestra: str, col_ids: list[str]) -> dict:
    print("\n── LLM layout ──────────────────────────────")

    raw = llamar_llm1(
        f"Analiza esta muestra:\n\n{muestra}",
        system=SYSTEM_LAYOUT
    )

    print("\nRespuesta LLM:\n", raw[:1000])

    # extraer JSON
    start = raw.find("{")
    end   = raw.rfind("}") + 1
    clean = raw[start:end]

    try:
        lr = json.loads(clean)
    except:
        print("❌ Error parseando JSON")
        return None

    return lr

## Celda 10 — Ejecución del Paso 1: recorrer todas las hojas

Itera sobre las hojas del Excel, filtra por contenido financiero con `es_hoja_financiera`, serializa una muestra de cada hoja válida y llama a `detectar_layout`. El resultado queda en `hojas_procesadas`, que asocia cada hoja con su layout detectado y sus datos en crudo.

El output muestra la muestra enviada al LLM y la respuesta recibida, útil para depurar casos donde el layout no se detecta correctamente.


In [20]:
hojas = leer_excel_multi(RUTA_EXCEL)

hojas_procesadas = {}

for nombre, (rows, col_ids) in hojas.items():

    if not es_hoja_financiera(rows):
        continue

    muestra = serializar_muestra(rows, col_ids)
    
    # 👇 añade esto
    print(f"\n── Mapeo hoja '{nombre}' ──────────────")
    print(muestra[:2000])

    # 🔥 AQUÍ ÚNICA llamada al LLM
    layout = detectar_layout(muestra, col_ids)

    if not layout:
        continue

    hojas_procesadas[nombre] = {
        "rows": rows,
        "col_ids": col_ids,
        "layout": layout
    }


── Mapeo hoja 'Page 1' ──────────────
fila=0, col=col_1: GRIFOLS SA
fila=2, col=col_0: Balance/Estado de resultados
fila=4, col=col_0: Cuentas No Consolidadas
fila=4, col=col_3: 31/12/2024
fila=4, col=col_5: 31/12/2023
fila=4, col=col_7: 31/12/2022
fila=5, col=col_3: mil EUR
fila=5, col=col_5: mil EUR
fila=5, col=col_7: mil EUR
fila=6, col=col_2: 12 meses
fila=6, col=col_4: 12 meses
fila=6, col=col_6: 12 meses
fila=7, col=col_2: Aprobado
fila=7, col=col_4: Aprobado
fila=7, col=col_6: Aprobado
fila=8, col=col_2: Normal PGC 2007
fila=8, col=col_4: Normal PGC 2007
fila=8, col=col_6: Normal PGC 2007
fila=9, col=col_0: Activo
fila=10, col=col_0: A) Activo no corriente
fila=10, col=col_2: 12497590
fila=10, col=col_4: 11416223
fila=10, col=col_6: 12647002
fila=11, col=col_0: I Inmovilizado intangible
fila=11, col=col_2: 23959
fila=11, col=col_4: 19941
fila=11, col=col_6: 23213
fila=12, col=col_0: 1. Fondo de comercio de consolidación
fila=12, col=col_2: n.d.
fila=12, col=col_4: n.d.
fila=12,

## Celda 11 — Paso 2: construcción del DataFrame de conceptos

`construir_df_conceptos` es la pieza que traduce el layout detectado por el LLM en datos tabulares concretos. Para cada hoja procesa el caso según el tipo de layout:

- **Layout vertical** (el más habitual en SABI): recorre las filas, extrae el texto de la columna de conceptos y los valores de las columnas de períodos. Maneja el caso de hojas sin cabecera de fechas heredando los períodos de la primera hoja que sí los tenga.
- **Layout horizontal** (Bloomberg, plantillas corporativas): recorre las columnas, extrae el texto de la fila de conceptos y los valores de las filas de períodos.

El resultado es un DataFrame con columnas `{sheet, idx, texto, fecha_1, fecha_2, ...}` que contiene todas las partidas del balance y la PyG con sus valores numéricos por período. Las filas duplicadas por hojas con periodos solapados se fusionan con `groupby().first()`.

> El `multiplicador` detectado por el LLM se ignora deliberadamente (fijado a 1). Delegar la conversión de unidades al modelo introducía errores en algunos casos; es más fiable mantener los valores en las unidades originales del fichero y documentarlo.


In [21]:
# ── PASO 2: construir DataFrame de conceptos con valores ─────────────────

def construir_df_conceptos(hojas_procesadas: dict) -> pd.DataFrame:
    todas_filas = []

    # ── detectar periodos canónicos: los de la primera hoja que los tenga reales ─
    periodos_canonicos = []
    for data in hojas_procesadas.values():
        p = data["layout"].get("periodos") or []
        if p and not any("periodo" in x.lower() for x in p):
            periodos_canonicos = p
            break

    print(f"📅 Periodos canónicos detectados: {periodos_canonicos}")

    for nombre, data in hojas_procesadas.items():
        rows   = data["rows"]
        layout = data["layout"]
        tipo   = layout.get("layout_tipo")
        #para quitarte movidas de las unidades, antes estaba esto puesto y daba cosas locas: mult = layout.get("multiplicador", 1) or 1
        mult = 1

        # usar periodos propios si son reales, si no heredar los canónicos
        periodos_raw      = layout.get("periodos") or []
        periodos_genericos = not periodos_raw or any("periodo" in x.lower() for x in periodos_raw)
        periodos           = periodos_canonicos if periodos_genericos else periodos_raw

        # ── HORIZONTAL: conceptos en una fila, valores en otras filas ────
        if tipo == "horizontal":
            fila_conceptos = layout.get("fila_conceptos")
            filas_valores  = layout.get("filas_valores") or []

            if fila_conceptos is None:
                print(f"⚠️  {nombre}: fila_conceptos no detectada, saltando")
                continue

            labels = periodos if len(periodos) == len(filas_valores) \
                     else [f"periodo_{i}" for i in range(len(filas_valores))]

            fila_conceptos_row = rows[fila_conceptos]

            for col_key, valor in fila_conceptos_row.items():
                texto = str(valor).strip()
                if not texto or texto.lower() in ("nan", "none", "nat", ""):
                    continue

                idx  = int(col_key.split("_")[1])
                fila = {"sheet": nombre, "idx": idx, "texto": texto}

                for label, fv in zip(labels, filas_valores):
                    v = rows[fv].get(col_key)
                    try:
                        fila[label] = float(v) * mult
                    except (TypeError, ValueError):
                        fila[label] = None

                todas_filas.append(fila)

        # ── VERTICAL: conceptos en una columna, valores en otras columnas ─
        elif tipo == "vertical":
            col_conceptos = layout.get("col_conceptos")
            cols_valores  = layout.get("cols_valores") or []
            filas_valores = layout.get("filas_valores") or []
            fila_inicio   = layout.get("fila_datos_inicio", 0)

            if col_conceptos is None:
                print(f"⚠️  {nombre}: col_conceptos no detectada, saltando")
                continue

            # cols_valores puede venir null en algunas hojas, fallback a filas_valores
            if not cols_valores and filas_valores:
                print(f"⚠️  {nombre}: cols_valores null, intentando con filas_valores como fallback")

            labels = periodos if len(periodos) == len(cols_valores) \
                     else [f"periodo_{i}" for i in range(len(cols_valores))]

            col_key = f"col_{col_conceptos}"

            for i, row in enumerate(rows):
                if i < fila_inicio:
                    continue
                texto = str(row.get(col_key, "")).strip()
                if not texto or texto.lower() in ("nan", "none", "nat", "n.d.", ""):
                    continue

                fila = {"sheet": nombre, "idx": i, "texto": texto}

                for label, cv in zip(labels, cols_valores):
                    v = row.get(f"col_{cv}")
                    try:
                        fila[label] = float(v) * mult
                    except (TypeError, ValueError):
                        fila[label] = None

                todas_filas.append(fila)

        else:
            print(f"⚠️  {nombre}: layout_tipo desconocido ({tipo})")

    df_conceptos = pd.DataFrame(todas_filas)

    # ── unificar columnas: quedarnos solo con sheet, idx, texto + periodos canónicos ─
    cols_fijas   = ["sheet", "idx", "texto"]
    cols_periodo = [c for c in df_conceptos.columns if c not in cols_fijas]

    # si hay columnas duplicadas de periodos (ej: "31/12/2024" aparece dos veces), fusionar
    df_conceptos = df_conceptos.groupby(["sheet", "idx", "texto"], as_index=False).first()

    # reordenar
    cols_orden = cols_fijas + [c for c in df_conceptos.columns if c not in cols_fijas]
    df_conceptos = df_conceptos[cols_orden]

    print(f"✅ DataFrame de conceptos: {len(df_conceptos)} filas, {len(df_conceptos.columns)} columnas")
    print(f"   Columnas: {list(df_conceptos.columns)}")
    return df_conceptos


df_conceptos = construir_df_conceptos(hojas_procesadas)
df_conceptos

📅 Periodos canónicos detectados: ['31/12/2024', '31/12/2023', '31/12/2022']
✅ DataFrame de conceptos: 161 filas, 6 columnas
   Columnas: ['sheet', 'idx', 'texto', '31/12/2024', '31/12/2023', '31/12/2022']


,sheet,idx,texto,31/12/2024,31/12/2023,31/12/2022
0,Page 1,10,A) Activo no corriente,12497590.0,11416223.0,12647002.0
1,Page 1,11,I Inmovilizado intangible,23959.0,19941.0,23213.0
2,Page 1,12,1. Fondo de comercio de consolidación,NaN,NaN,NaN
3,Page 1,13,2. Investigación,NaN,NaN,NaN
4,Page 1,14,3. Propiedad intelectual,NaN,NaN,NaN
...,...,...,...,...,...,...
156,Page 1,178,B) Operaciones interrumpidas,NaN,NaN,NaN
157,Page 1,179,25. Resultado del ejercicio procedente de oper...,NaN,NaN,NaN
158,Page 1,180,A5) Resultado del ejercicio (A4 + 25),-83138.0,-246735.0,-266296.0
159,Page 1,182,Resultado atribuido a la sociedad dominante,NaN,NaN,NaN


## Celda 12 — Paso 3: selección de las 22 partidas mediante LLM con índice posicional

Esta es la pieza más importante del pipeline para la robustez ante formatos heterogéneos.

Se construye un índice `sheet=X, idx=Y: texto_partida` con todas las partidas del DataFrame en orden y se envía al LLM. El prompt instruye al modelo para que:

- **Razone por proximidad de fila** respecto a cabeceras de sección para resolver partidas duplicadas (ej: "Deudas con entidades de crédito" aparece igual en LP y en CP — solo la posición relativa respecto a anclas como "Pasivo no corriente" o "Pasivo corriente" permite distinguirlas).
- **Entienda la estructura acumulativa de la PyG**: no confundir "A1) Resultado de explotación (1+2+...+14)" con "5. Otros ingresos de explotación" aunque ambos contengan las mismas palabras clave.
- **Maneje denominaciones no estándar**: inglés, abreviaturas, grafías con tildes o sin ellas.

Por qué este enfoque en lugar de embeddings o fuzzy matching: ambos métodos producen fallos sistemáticos en partidas con nombres similares en secciones distintas del balance, y necesitarían un reranker adicional que en la práctica equivale a hacer exactamente esto de forma menos directa. El experimento comparativo está documentado en la memoria (sección 4.2.2).

El output es un JSON `{alias: {sheet, idx, partida}}` con los 22 mapeos.


In [22]:
# ── PASO 3: LLM selecciona partidas ──────────────────────────────────────

# incluir sheet para que el LLM no se lo invente
lineas_partidas = []
for _, row in df_conceptos.iterrows():
    lineas_partidas.append(f"sheet={row['sheet']}, idx={int(row['idx'])}: {row['texto']}")
partidas_formateadas = "\n".join(lineas_partidas)

prompt_mapeo = f"""Eres un experto en contabilidad española (PGC 2007).
Tienes la lista completa de partidas de un balance financiero extraído de un Excel,
en orden exacto tal como aparecen, con su hoja e índice.

PARTIDAS DISPONIBLES:
{partidas_formateadas}

INSTRUCCIONES GENERALES:
- Devuelve EXACTAMENTE el texto de la partida tal como aparece en la lista.
- Si una partida no existe en este balance devuelve null.

INSTRUCCIONES ESPECIALES:
- 'patrimonio_neto': Puede aparecer como Patrimonio neto o todo junto como patrimonio neto o con letras o numeros antes o despues.
- 'fondos_propios': si no hay partida especifica de fondos propios devuelve patrimonio neto para los dos pero si hay fondos propios devuelve la partida específica.
- 'pasivo_no_corriente': puede aparecer como 'B) Pasivo no corriente'.
- 'pasivo_corriente': puede aparecer como 'C) Pasivo corriente'.
- 'total_activo': puede aparecer como 'Total activo (A + B)'.
- 'activo_corriente': puede aparecer como 'B) Activo corriente'.
- 'resultado_ejercicio': puede aparecer como 'A5) Resultado del ejercicio (A4 + 25)'.
- 'efectivo': puede aparecer como 'VII Efectivo y otros activos liquidos equivalentes'.
- 'deuda_credito_lp' y 'deuda_credito_cp': busca la partida "Deudas con entidades de credito"
  (puede aparecer dos veces en la lista).
  Una vez localizada, mira los indices inmediatamente anteriores y determina
  si esta ENTRE o MAS CERCA de alguna de estas anclas de CP:
  "Deudas a corto plazo", "corto plazo", "pasivo corriente"
  o de estas anclas de LP:
  "Deudas a largo plazo", "largo plazo", "pasivo no corriente".
  La ancla que aparezca mas cerca en numero de indices determina si es LP o CP.
  Si es CP → asigna a deuda_credito_cp y pon null en deuda_credito_lp.
  Si es LP → asigna a deuda_credito_lp y pon null en deuda_credito_cp.
  IMPORTANTE: "Deudas a corto plazo" y "Deudas a largo plazo" tambien cuentan como anclas,
  no solo "pasivo corriente" y "pasivo no corriente".

Ejemplo de output:
{{
  "patrimonio_neto":            {{"sheet": "Tabla1", "idx": 40, "partida": "A) Patrimonio neto"}},
  "fondos_propios":             {{"sheet": "Tabla1", "idx": 40, "partida": "Fondos propios"}},
  "pasivo_no_corriente":        {{"sheet": "Tabla1", "idx": 61, "partida": "B) Pasivo no corriente"}},
  "deudas_lp":                  {{"sheet": "Tabla1", "idx": 63, "partida": "II Deudas a largo plazo"}},
  "deudas_cp":                  {{"sheet": "Tabla1", "idx": 80, "partida": "III Deudas a corto plazo"}},
  "pasivo_corriente":           {{"sheet": "Tabla1", "idx": 75, "partida": "C) Pasivo corriente"}},
  "total_activo":               {{"sheet": "Tabla1", "idx": 38, "partida": "Total activo (A + B)"}},
  "activo_corriente":           {{"sheet": "Tabla1", "idx": 21, "partida": "B) Activo corriente"}},
  "existencias":                null,
  "deudores_comerciales":       {{"sheet": "Tabla1", "idx": 24, "partida": "III Deudores comerciales y otras cuentas a cobrar"}},
  "efectivo":                   {{"sheet": "Tabla1", "idx": 37, "partida": "VII Efectivo y otros activos liquidos equivalentes"}},
  "inmovilizado_material":      {{"sheet": "Tabla1", "idx": 8,  "partida": "II Inmovilizado material"}},
  "acreedores_comerciales":     {{"sheet": "Tabla1", "idx": 88, "partida": "V Acreedores comerciales y otras cuentas a pagar"}},
  "cifra_negocios":             {{"sheet": "Tabla1", "idx": 100, "partida": "1. Importe neto de la cifra de negocios"}},
  "otros_ingresos_explotacion": {{"sheet": "Tabla1", "idx": 110, "partida": "5. Otros ingresos de explotacion"}},
  "resultado_explotacion":      {{"sheet": "Tabla1", "idx": 132, "partida": "A1) Resultado de explotacion (1 + 2 + 3 + 4 + 5 + 6 + 7 + 8 + 9 + 10 + 11 + 12 + 13 + 14)"}},
  "amortizacion":               {{"sheet": "Tabla1", "idx": 121, "partida": "8. Amortizacion del inmovilizado"}},
  "gastos_financieros":         {{"sheet": "Tabla1", "idx": 137, "partida": "16. Gastos financieros"}},
  "resultado_antes_impuestos":  {{"sheet": "Tabla1", "idx": 155, "partida": "A3) Resultado antes de impuestos (A1 + A2+21+22+23)"}},
  "resultado_ejercicio":        {{"sheet": "Tabla1", "idx": 160, "partida": "A5) Resultado del ejercicio (A4 + 25)"}},
  "deuda_credito_lp":           null,
  "deuda_credito_cp":           {{"sheet": "Tabla1", "idx": 82, "partida": "2. Deudas con entidades de credito"}}
}}

Ahora analiza las partidas disponibles de este balance y devuelve el JSON con los valores reales. SOLO JSON sin markdown.
"""

response = llamar_llm2(prompt_mapeo, system="Devuelve solo JSON sin markdown.")
print(f"Input aprox tokens: {len(prompt_mapeo.split())}")
print(response)

raw = response.strip().strip("```json").strip("```").strip()

try:
    mapeo_llm = json.loads(raw)
    print("\n── Mapeo LLM ──")
    for alias, info in mapeo_llm.items():
        if info:
            print(f"  {alias}: sheet={info['sheet']}, idx={info['idx']} → {info['partida']}")
        else:
            print(f"  {alias}: null")
except json.JSONDecodeError:
    print("⚠️ No devolvió JSON limpio")
    print(raw)

Input aprox tokens: 2054
```json
{
  "patrimonio_neto": {"sheet": "Page 1", "idx": 50, "partida": "A) Patrimonio neto"},
  "fondos_propios": {"sheet": "Page 1", "idx": 51, "partida": "A-1) Fondos propios"},
  "pasivo_no_corriente": {"sheet": "Page 1", "idx": 75, "partida": "B) Pasivo no corriente"},
  "deudas_lp": {"sheet": "Page 1", "idx": 79, "partida": "2. Deudas con entidades de crédito"},
  "deudas_cp": {"sheet": "Page 1", "idx": 97, "partida": "2. Deudas con entidades de crédito"},
  "pasivo_corriente": {"sheet": "Page 1", "idx": 90, "partida": "C) Pasivo corriente"},
  "total_activo": {"sheet": "Page 1", "idx": 47, "partida": "Total activo (A + B)"},
  "activo_corriente": {"sheet": "Page 1", "idx": 30, "partida": "B) Activo corriente"},
  "existencias": {"sheet": "Page 1", "idx": 32, "partida": "II Existencias"},
  "deudores_comerciales": {"sheet": "Page 1", "idx": 33, "partida": "III Deudores comerciales y otras cuentas a cobrar"},
  "efectivo": {"sheet": "Page 1", "idx": 46, "

## Celda 13 — Paso 4: extracción de valores por período

Con el mapeo del LLM se cruzan los índices `(sheet, idx)` con el DataFrame de conceptos para recuperar el valor numérico de cada partida en cada fecha. El resultado es `valores_por_fecha`: un diccionario `{fecha: {alias: valor}}` listo para el cálculo de ratios.

Este paso es completamente determinista: el LLM ya ha hecho el trabajo de identificación, aquí solo se hace un lookup posicional en el DataFrame.


In [23]:
# ── PASO 4: extraer valores por periodo ──────────────────────────────────

cols_periodo = [c for c in df_conceptos.columns if c not in ("sheet", "idx", "texto")]
valores_por_fecha = {}

for alias, info in mapeo_llm.items():
    if info is None:
        continue

    fila = df_conceptos[
        (df_conceptos["sheet"] == info["sheet"]) &
        (df_conceptos["idx"]   == info["idx"])
    ]

    if fila.empty:
        print(f"⚠️ {alias}: sheet={info['sheet']} idx={info['idx']} no encontrado")
        continue

    for periodo in cols_periodo:
        if periodo not in valores_por_fecha:
            valores_por_fecha[periodo] = {}
        if alias not in valores_por_fecha[periodo]:
            v = fila.iloc[0][periodo]
            valores_por_fecha[periodo][alias] = None if pd.isna(v) else v

print(pd.DataFrame(valores_por_fecha).T)

            patrimonio_neto  fondos_propios  pasivo_no_corriente  deudas_lp  \
31/12/2024        1953852.0       1972243.0           10572251.0   901345.0   
31/12/2023        2101487.0       2044735.0           10561625.0  1308026.0   
31/12/2022        2343125.0       2285248.0           10323483.0  1340473.0   

            deudas_cp  pasivo_corriente  total_activo  activo_corriente  \
31/12/2024    29383.0          313933.0    12840036.0          342446.0   
31/12/2023    68542.0          298105.0    12961217.0         1544994.0   
31/12/2022    60899.0          226493.0    12893101.0          246099.0   

            existencias  deudores_comerciales  ...  acreedores_comerciales  \
31/12/2024      13342.0               79885.0  ...                 94624.0   
31/12/2023      12333.0               79873.0  ...                112436.0   
31/12/2022      11439.0               72869.0  ...                 82987.0   

            cifra_negocios  otros_ingresos_explotacion  resultado_exp

## Celda 14 — Paso 5: cálculo de 22 ratios financieros

Calcula los 22 ratios agrupados en 5 categorías a partir de `valores_por_fecha`:

| Categoría | Ratios |
|---|---|
| **Solvencia** | Ratio de solvencia, autonomía financiera, endeudamiento, DFN/FP |
| **Cobertura** | DFN/EBITDA, ICR (EBITDA/gastos financieros), carga financiera/CN |
| **Liquidez** | Ratio corriente, ácido, fondo de maniobra/CN, tesorería/activo |
| **Rentabilidad** | Margen EBITDA, margen EBIT, margen neto, ROA, ROE |
| **Eficiencia operativa** | PMC, PMP, días de stock, ciclo de caja (CCC) |

Notas de implementación:
- `safe_div` evita errores por denominador nulo o `None` en cualquier ratio.
- `EBITDA = EBIT - amortización`: la amortización en SABI viene sin signo (valor absoluto), por lo que la resta es directa.
- `DFN = deuda financiera total - efectivo`, donde la deuda financiera es la suma de créditos bancarios a LP y CP.
- `gastos_financieros` se toma en valor absoluto con `abs()` porque en la PyG aparece como gasto negativo.
- El CCC se calcula como `PMC + días_stock - PMP`.


In [24]:
# ── PASO 5: calcular ratios ───────────────────────────────────────────────

def safe_div(a, b):
    if a is None or b is None or b == 0: return None
    return round(a / b, 6)

def pct(a, b):   r = safe_div(a, b); return round(r * 100, 2) if r is not None else None
def ratio(a, b): r = safe_div(a, b); return round(r, 2)       if r is not None else None
def dias(a, b):  r = safe_div(a, b); return round(r * 365, 1) if r is not None else None

ratios_calculados = {}

for fecha, v in valores_por_fecha.items():
    cn    = v.get('cifra_negocios')
    oi    = v.get('otros_ingresos_explotacion') or 0
    ebit  = v.get('resultado_explotacion')
    amort = v.get('amortizacion') or 0
    gf    = v.get('gastos_financieros')
    res   = v.get('resultado_ejercicio')
    act   = v.get('total_activo')
    actc  = v.get('activo_corriente')
    exst  = v.get('existencias') or 0
    deud  = v.get('deudores_comerciales')
    efec  = v.get('efectivo') or 0
    fp    = v.get('fondos_propios')
    pasc  = v.get('pasivo_corriente')
    acr   = v.get('acreedores_comerciales')
    delp  = v.get('deuda_credito_lp') or 0
    decp  = v.get('deuda_credito_cp') or 0
    pnc   = v.get('pasivo_no_corriente')

    ebitda  = (ebit - amort) if ebit  is not None else None
    gf_abs  = abs(gf)        if gf    is not None else None
    pas_tot = (pnc + pasc)   if (pnc  is not None and pasc is not None) else None
    fm      = (actc - pasc)  if (actc is not None and pasc is not None) else None
    deuda_f = delp + decp
    dfn     = deuda_f - efec

    pmc   = dias(deud, cn)
    pmp   = dias(acr,  cn)
    stock = dias(exst, cn)
    ccc   = round((pmc or 0) + (stock or 0) - (pmp or 0), 1) if all(x is not None for x in [pmc, pmp, stock]) else None

    ratios_calculados[fecha] = {
        "SOL01_solvencia":              ratio(act, pas_tot),
        "SOL02_autonomia_financiera":   pct(fp, act),
        "SOL03_endeudamiento":          ratio(pas_tot, fp),
        "SOL04_dfn_sobre_fp":           pct(dfn, fp),
        "COV01_dfn_sobre_ebitda":       ratio(dfn, ebitda),
        "COV02_icr":                    ratio(ebitda, gf_abs),
        "COV03_carga_financiera_cn":    pct(gf_abs, cn),
        "LIQ01_ratio_corriente":        ratio(actc, pasc),
        "LIQ02_ratio_acido":            ratio((actc - exst) if actc is not None else None, pasc),
        "LIQ03_fm_sobre_cn":            pct(fm, cn),
        "LIQ04_tesoreria_sobre_activo": pct(efec, act),
        "REN01_margen_ebitda":          pct(ebitda, cn),
        "REN02_margen_ebit":            pct(ebit, cn),
        "REN03_margen_neto":            pct(res, cn),
        "REN04_roa":                    pct(ebit, act),
        "REN05_roe":                    pct(res, fp),
        "APL01_deuda_fin_sobre_activo": pct(deuda_f, act),
        "APL02_pasivo_sobre_activo":    pct(pas_tot, act),
        "EFI01_pmc":   pmc,
        "EFI02_pmp":   pmp,
        "EFI03_stock": stock,
        "EFI04_ccc":   ccc,
    }

df_ratios = pd.DataFrame(ratios_calculados).T

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)
display(df_ratios)
pd.reset_option('display.max_columns')
pd.reset_option('display.max_rows')
pd.reset_option('display.width')
pd.reset_option('display.float_format')

,SOL01_solvencia,SOL02_autonomia_financiera,SOL03_endeudamiento,SOL04_dfn_sobre_fp,COV01_dfn_sobre_ebitda,COV02_icr,COV03_carga_financiera_cn,LIQ01_ratio_corriente,LIQ02_ratio_acido,LIQ03_fm_sobre_cn,LIQ04_tesoreria_sobre_activo,REN01_margen_ebitda,REN02_margen_ebit,REN03_margen_neto,REN04_roa,REN05_roe,APL01_deuda_fin_sobre_activo,APL02_pasivo_sobre_activo,EFI01_pmc,EFI02_pmp,EFI03_stock,EFI04_ccc
31/12/2024,1.18,15.36,5.52,47.13,1.74,0.88,86.40,1.09,1.05,4.07,0.01,76.35,74.02,-11.86,4.04,-4.22,7.25,84.78,41.60,49.30,6.90,-0.80
31/12/2023,1.19,15.78,5.31,66.69,6.13,0.41,86.77,5.18,5.14,201.36,0.10,35.95,33.16,-39.84,1.58,-12.07,10.62,83.79,47.10,66.30,7.30,-11.90
31/12/2022,1.22,17.72,4.62,60.72,18.94,0.18,82.27,1.09,1.04,4.01,0.11,15.00,12.06,-54.50,0.46,-11.65,10.87,81.83,54.40,62.00,8.50,0.90
